# 05 — Vendo a fraude: visualização com neo4j-viz

Nos notebooks anteriores você **descobriu** a fraude lendo tabelas de resultado.
Agora vamos **vê-la**. Fraude é literalmente um problema de *forma* — anéis,
estrelas, funis — e essas formas saltam aos olhos num desenho muito antes de
aparecerem numa lista de linhas.

Usamos o [`neo4j-viz`](https://pypi.org/project/neo4j-viz/), biblioteca oficial da
Neo4j que renderiza um grafo interativo direto no notebook, a partir do resultado
de uma query Cypher comum.

> Pré-requisito: ter rodado o notebook `02` (carga) e o `04` até pelo menos a
> seção 5 (que cria a marcação `:Suspeito` e o relacionamento
> `TRANSFERIU_PIX_PARA`).

## Instalação — e por que a versão está fixada

Note o `<1.7` no `neo4j-viz`. Não é capricho: a versão 1.7.0 usa
`traitlets.Instance[...]` (subscrição genérica), que exige um `traitlets` mais novo
do que o pré-instalado no Google Colab. O resultado é este erro no import:

```
TypeError: type 'Instance' is not subscriptable
```

A `1.6.0` tem exatamente a mesma API que usamos aqui e não depende desse recurso,
então funciona no Colab sem mais nenhum ajuste.

In [ ]:
!pip install -q "neo4j-viz[neo4j]<1.7" neo4j-rust-ext python-dotenv

A célula abaixo descarta da memória qualquer versão do `neo4j-viz` carregada antes,
para o próximo import usar a que acabou de ser instalada. Assim você não precisa
reiniciar a sessão.

In [ ]:
import importlib
import sys

removidos = [m for m in list(sys.modules) if m.startswith("neo4j_viz")]
for modulo in removidos:
    del sys.modules[modulo]
importlib.invalidate_caches()

if removidos:
    print(f"{len(removidos)} módulos de uma versão anterior removidos da memória.")
    print("O próximo import vai ler a versão recém-instalada.")
else:
    print("Nenhuma versão anterior carregada nesta sessão — nada a limpar.")

### Teste de fumaça

Em vez de conferir números de versão, a célula abaixo **exercita a biblioteca** num
grafo de dois nós, passando exatamente pelo caminho que o `from_neo4j` usa. Se algo
estiver incompatível, o erro aparece aqui — com instrução de como resolver — e não
no meio da visualização.

In [ ]:
import importlib.metadata as metadata

print(f"neo4j-viz {metadata.version('neo4j-viz')}")

try:
    from neo4j_viz import Node, Relationship, VisualizationGraph

    _teste = VisualizationGraph(
        nodes=[Node(id="a", caption="Alfa"), Node(id="b", caption="Beta")],
        relationships=[Relationship(id="r", source="a", target="b", caption="LIGA")],
    )
    _teste.color_nodes(field="caption")   # mesmo caminho interno do from_neo4j
    _teste.render()                       # e o de renderização
    print("Biblioteca operacional ✅")
except Exception as e:
    print(f"\n❌ {type(e).__name__}: {e}\n")
    print("Reinicie a sessão (Ambiente de execução > Reiniciar sessão)")
    print("e rode o notebook desde a primeira célula.")

## Credenciais

Este notebook busca as credenciais em três lugares, na ordem:

1. **Secrets do Colab** — o ícone de chave 🔑 na barra lateral esquerda.
2. **Variáveis de ambiente** — incluindo um arquivo `.env` na pasta do projeto
   (copie o `.env.example` e preencha). É o caminho para quem roda localmente.
3. **Pergunta na tela** — se não achou nas opções acima, pergunta aqui mesmo.

> ⚠️ **No Colab, cada notebook precisa de permissão para cada secret.** Ter criado
> o secret na sua conta não basta: abra o painel 🔑 e ative a chave
> **"Acesso ao notebook"** (*Notebook access*) para **este** notebook. Sem isso o
> secret é ignorado silenciosamente e o notebook volta a perguntar na tela.
>
> Se os secrets estiverem configurados (e liberados), a célula abaixo não pergunta
> nada — apenas conecta.

Além de poupar digitação, as duas primeiras opções evitam que a URI da sua
instância fique gravada na saída da célula caso você comite o notebook.

In [ ]:
import os
from getpass import getpass

try:  # carrega um arquivo .env, se existir
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass  # python-dotenv não instalado, ou não há .env — segue o baile


def credencial(nome, prompt, secreta=False, padrao=None):
    """Busca em: Secrets do Colab > variável de ambiente (.env) > pergunta na tela."""
    try:
        from google.colab import userdata
        if valor := userdata.get(nome):
            return valor
    except ImportError:
        pass  # não estamos no Colab
    except Exception as e:
        # O caso confuso: o secret existe, mas este notebook não tem permissão.
        # Sem este aviso, o notebook só voltaria a perguntar, sem explicar por quê.
        if "NotebookAccess" in type(e).__name__:
            print(f"⚠️  O secret '{nome}' existe, mas este notebook não tem acesso a ele.")
            print(f"    Abra o painel 🔑 e ative 'Acesso ao notebook' para '{nome}'.")

    if valor := os.environ.get(nome):
        return valor

    return (getpass(prompt) if secreta else input(prompt)).strip() or padrao


NEO4J_URI = credencial("NEO4J_URI", "URI do Neo4j (ex.: neo4j+s://xxxx.databases.neo4j.io): ")
NEO4J_USER = credencial("NEO4J_USERNAME", "Usuário [neo4j]: ", padrao="neo4j")
NEO4J_PASSWORD = credencial("NEO4J_PASSWORD", "Senha: ", secreta=True)
# Atenção: em instâncias AuraDB recentes o banco NÃO se chama "neo4j", e sim o
# próprio instance id (o prefixo da URI). Confira em Aura Console > sua instância,
# ou rode SHOW DATABASES.
NEO4J_DATABASE = credencial("NEO4J_DATABASE", "Nome do banco (geralmente = instance id) [neo4j]: ", padrao="neo4j")

In [ ]:
from neo4j_viz.neo4j import from_neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Conectado!")

## 1. A visão 360° de um cliente

Vamos começar pequeno: **um** cliente, com tudo que o grafo sabe sobre ele — seus
identificadores (RG, e-mail, telefone) e todas as transações que fez, cada uma
apontando para seu destino.

`from_neo4j` aceita diretamente o que `driver.execute_query(...)` retorna e já
colore cada nó por label, com legenda. Repare que essa única imagem contém **todo
o modelo** que você desenhou no notebook 02.

In [ ]:
# Escolhemos um cliente de verdade do banco — os CPFs são gerados
# aleatoriamente no notebook 01, então não dá para fixar um valor aqui.
registros, _, _ = driver.execute_query(
    """
    MATCH (c:Cliente)-[:REALIZOU]->(:Transacao)
    RETURN c.cpf AS cpf, c.nome AS nome, count(*) AS transacoes
    ORDER BY transacoes DESC
    LIMIT 1
    """,
    database_=NEO4J_DATABASE,
)
CPF_EXEMPLO = registros[0]["cpf"]
print(f"Cliente escolhido: {registros[0]['nome']} (CPF {CPF_EXEMPLO}, {registros[0]['transacoes']} transações)")

resultado = driver.execute_query(
    """
    MATCH (c:Cliente {cpf: $cpf})
    OPTIONAL MATCH p1=(c)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]->()
    OPTIONAL MATCH p2=(c)-[:REALIZOU]->(:Transacao)-[:PARA]->()
    RETURN p1, p2 LIMIT 60
    """,
    cpf=CPF_EXEMPLO,
    database_=NEO4J_DATABASE,
)

VG = from_neo4j(resultado)
VG.render()

> **Duas dicas de uso:**
>
> 1. O desenho pode aparecer **vazio por 1 ou 2 segundos** — o layout é uma
>    simulação física que precisa de um instante para se acomodar. Espere antes de
>    achar que deu errado.
> 2. Se depois disso ainda ficar vazio ou muito espalhado, clique no botão
>    **zoom to fit** (ícone de expandir, canto inferior direito) para enquadrar
>    todos os nós. É comum quando o resultado tem muitos grupos desconectados.

## 2. Os anéis de fraude

Aqui está o primeiro achado do notebook 04 em forma de imagem: os clientes
marcados como `:Suspeito` e os identificadores que eles compartilham.

Uma lista de pares de `cpf` não deixa óbvio o que está acontecendo. O
desenho deixa: cada **estrela** é um anel de fraude, e o nó no centro é o
RG/e-mail/telefone que várias "identidades" diferentes estão reaproveitando.

In [ ]:
resultado = driver.execute_query(
    """
    MATCH p=(c:Cliente:Suspeito)-[:TEM_RG|TEM_EMAIL|TEM_TELEFONE]-(id)
    RETURN p
    """,
    database_=NEO4J_DATABASE,
)

if resultado.records:
    VG = from_neo4j(resultado)
    VG.render()
else:
    print("Nenhum cliente :Suspeito encontrado — rode a seção 3 do notebook 04 primeiro.")

## 3. A rede de lavagem via Pix

O segundo achado: fraudadores (`:Suspeito`) e as contas que receberam Pix deles.

Filtramos por `valor > 20000` de propósito. Sem filtro, o desenho mistura as
transferências fraudulentas com todo o Pix legítimo que esses clientes também
fizeram — e vira um emaranhado ilegível. Com o filtro, sobra o padrão real: poucos
fraudadores, várias contas de destino, e **várias setas convergindo nas mesmas
contas** — o formato clássico de uma rede de lavagem.

In [ ]:
resultado = driver.execute_query(
    """
    MATCH p=(:Cliente:Suspeito)-[r:TRANSFERIU_PIX_PARA]->(:Cliente)
    WHERE r.valor > 20000
    RETURN p
    """,
    database_=NEO4J_DATABASE,
)

if resultado.records:
    VG = from_neo4j(resultado)
    # initial_zoom afasta a câmera para caber a rede inteira já na primeira exibição
    VG.render(initial_zoom=0.5)
else:
    print("Nenhum relacionamento TRANSFERIU_PIX_PARA encontrado — rode a seção 5 do notebook 04 primeiro.")

Experimente baixar o corte para `> 5000` e rodar de novo: o sinal some no
ruído. Escolher o corte certo é parte do trabalho de quem investiga fraude — e o
grafo é justamente a ferramenta que deixa esse ajuste barato de testar.

## 4. Destacando os alvos principais

Um último ajuste útil numa investigação: **tamanho do nó proporcional ao valor
recebido**. Assim a imagem não só mostra quem está na rede, mas quem é o alvo
principal do dinheiro.

In [ ]:
resultado = driver.execute_query(
    """
    MATCH p=(:Cliente:Suspeito)-[r:TRANSFERIU_PIX_PARA]->(destino:Cliente)
    WHERE r.valor > 20000
    RETURN p
    """,
    database_=NEO4J_DATABASE,
)

# quanto cada conta recebeu no total (para dimensionar os nós)
recebido, _, _ = driver.execute_query(
    """
    MATCH (:Cliente:Suspeito)-[r:TRANSFERIU_PIX_PARA]->(destino:Cliente)
    WHERE r.valor > 20000
    RETURN destino.cpf AS cpf, sum(r.valor) AS total
    """,
    database_=NEO4J_DATABASE,
)
total_por_cliente = {r["cpf"]: r["total"] for r in recebido}

if resultado.records:
    VG = from_neo4j(resultado)
    for node in VG.nodes:
        cid = node.properties.get("cpf")
        node.caption = cid
        node.size = 10 + (total_por_cliente.get(cid, 0) / 15000)
    VG.render(initial_zoom=0.5)
else:
    print("Rode a seção 5 do notebook 04 primeiro.")

## Recapitulando

`neo4j-viz` transforma qualquer resultado de `driver.execute_query(...)` num grafo
interativo, sem exportar nada para outra ferramenta. Três coisas que valem levar
deste notebook:

1. **Filtrar é parte de visualizar.** Um grafo sem corte vira emaranhado; o corte
   certo revela o padrão.
2. **A forma é o achado.** Estrela = identificador reaproveitado. Funil = rede de
   lavagem. Você aprende a reconhecer isso de olho.
3. Para grafos maiores, ajuste o `LIMIT` das queries e o `max_allowed_nodes` de
   `.render(...)` (padrão: 10.000 nós).

### Para ir além

- [Documentação do neo4j-viz](https://pypi.org/project/neo4j-viz/)
- `neo4j_viz.gds.from_gds(...)` — visualiza direto um grafo projetado numa sessão
  de GDS/Aura Graph Analytics, sem precisar escrever o resultado no banco antes
- [Neo4j Bloom](https://neo4j.com/docs/bloom-user-guide/current/) — ferramenta
  visual de investigação, para quem não quer escrever Cypher

In [ ]:
driver.close()